In [12]:
!pip install termcolor


[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# =====================================================
# ResumeAtlas – NER Inference Demonstration Script
# =====================================================

from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import json, torch, os

# ------------------------------
# 1️⃣ Load Fine-Tuned Resume Model
# ------------------------------
model_path = "artifacts/ner_classifier_final"
print(f"\n🚀 Loading ResumeAtlas NER model from: {model_path}")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# ------------------------------
# 2️⃣ Load label mapping (43 career domains)
# ------------------------------
label_map_path = os.path.join(model_path, "label_map.json")

if os.path.exists(label_map_path):
    with open(label_map_path) as f:
        id2label = json.load(f)
    model.config.id2label = {int(k): v for k, v in id2label.items()}
    model.config.label2id = {v: int(k) for k, v in id2label.items()}
    print(f"✅ Loaded {len(id2label)} resume labels from label_map.json\n")
else:
    print("⚠️ label_map.json missing. Using fallback labels.\n")

# ------------------------------
# 3️⃣ Create pipeline (aggregated)
# ------------------------------
resume_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

# ------------------------------
# 4️⃣ Sample Resume-Based Text (realistic data)
# ------------------------------
resume_entries = [
    # Data Science professional resume snippet
    """Rohan Gupta — Data Scientist with 4+ years of experience in predictive modeling,
    statistical analysis, and developing machine learning systems using Python, TensorFlow, and SQL.
    Previously worked at Accenture as Senior Analyst handling financial forecasting models.""",

    # Software engineering focused resume snippet
    """Priya Sharma — Full Stack Developer specializing in ReactJS and Node.js.
    Experienced in designing RESTful APIs, deploying applications on AWS,
    and integrating microservices to deliver scalable web solutions.""",

    # DevOps and Cloud resume snippet
    """Kunal Mehta — DevOps Engineer experienced with Kubernetes, Docker, and CI/CD pipelines.
    Skilled in optimizing deployment processes, managing AWS infrastructure, and ensuring 99.9% system uptime.""",

    # Finance domain resume snippet
    """Anjali Verma — Finance Associate with a demonstrated history in financial reporting,
    auditing, and data-driven investment analysis. Led risk evaluation projects at Deloitte and improved turnover efficiency by 18%."""
]


print("[INFO] Running ResumeAtlas NER inference...\n")

# ------------------------------
# 5️⃣ Helper for cleaning tokens
# ------------------------------
def clean(word):
    return word.replace("##", "").strip()

# ------------------------------
# 6️⃣ Run inference
# ------------------------------
for i, resume in enumerate(resume_entries, 1):
    print(f"🟢 Resume Sample {i}:")
    print("-" * 90)
    results = resume_pipeline(resume)

    if not results:
        print("No entities detected.\n")
        continue

    print(f"{'Extracted Text':<35} | {'Predicted Label':<25} | {'Confidence':<10}")
    print("-" * 90)
    for ent in results:
        text = clean(ent.get("word", ""))
        label = ent.get("entity_group", "")
        score = f"{ent.get('score', 0):.3f}"
        print(f"{text:<35} | {label:<25} | {score:<10}")
    print("\n")

# ------------------------------
# 7️⃣ Display runtime context
# ------------------------------
device = "CUDA GPU" if torch.cuda.is_available() else "CPU"
print("System Info:")
print(f" - Execution Device: {device}")
print(f" - Model Source Path: {model_path}")
print("✅ ResumeAtlas NER Inference Completed Successfully!\n")



🚀 Loading ResumeAtlas NER model from: artifacts/ner_classifier_final
✅ Loaded 43 resume labels from label_map.json

[INFO] Running ResumeAtlas NER inference...

🟢 Resume Sample 1:
------------------------------------------------------------------------------------------
Extracted Text                      | Predicted Label           | Confidence
------------------------------------------------------------------------------------------
rohan                               | Architecture              | 0.038     
gupta                               | Banking                   | 0.042     
—                                   | Data Science              | 0.033     
data                                | Banking                   | 0.040     
scientist with                      | Agriculture               | 0.049     
4                                   | Architecture              | 0.039     
+                                   | Advocate                  | 0.040     
years                

In [22]:
from transformers import pipeline

# Use a real resume NER model (trained on entity tags)
ner_pipeline = pipeline(
    "token-classification",
    model="yashpwr/resume-ner-bert-v2",  # or any suitable resume model
    aggregation_strategy="simple"
)

text = "Yash Pandey is a Web Developer at Google in India."
results = ner_pipeline(text)
for ent in results:
    print(f"{ent['entity_group']}: {ent['word']} (score: {ent['score']:.2f})")


config.json: 0.00B [00:00, ?B/s]

C:\Users\ISHA PANDEY\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ISHA PANDEY\.cache\huggingface\hub\models--yashpwr--resume-ner-bert-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falli

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Designation: a Web Developer at (score: 0.70)
Companies worked at: Google in (score: 0.43)
